# 리랭커 모델 4종 비교 — BGE-reranker-v2-m3 / Qwen3-Reranker / ko-reranker / bge-reranker-v2-m3-ko

`agent/mapping/reranker.py`의 3단계(재정렬) 자리에 아직 실제 리랭커가 붙어있지 않습니다
(`rerank_scores()`가 `HCX_API_KEY` 미설정 시 `None`을 반환하고 항등 정렬로 폴백하는 상태, `reranker.py:44-65`).
이 노트북은 그 자리에 넣을 오픈소스 cross-encoder 리랭커 후보 4개를 비교합니다.

**리랭커는 임베딩과 달리 전체 코퍼스를 다시 훑지 않습니다.** 실제 파이프라인에서는
`keyword_search` + `embedding_search`가 추린 후보 목록만 재정렬합니다(`reranker.py:146-160`
`search_and_rerank()` 참고). 그래서 이 노트북에서도 `kosis_large_corpus.json`(표 1053개) 전체에
cross-encoder를 돌리는 대신, 문자 n-gram TF-IDF로 쿼리별 "헷갈리는 후보 shortlist"(상위 20개 + 정답
강제 포함)를 먼저 만들고, 그 shortlist 안에서 각 리랭커가 얼마나 정답을 상위로 끌어올리는지를 비교합니다.
(`kosis_large_corpus.json`에는 `keyword_search.py`가 쓰는 `keywords`/`title` 필드가 없어서
실제 keyword_search 로직 대신 TF-IDF로 그 역할을 대체합니다.)

**평가 방법**: `embedding_model_comparison.ipynb`와 동일한 50개 라벨링 테스트 문장
(`agent/mapping/test_mapping.py`의 원래 12개 + 변별력 강화용 38개)을 그대로 재사용합니다.
같은 벤치마크라 두 노트북의 결과를 나란히 비교할 수 있습니다.

**비교 대상 4개 모델:**
1. **BGE-reranker-v2-m3** (`BAAI/bge-reranker-v2-m3`, ~568M) — 다국어 범용, 임베딩 비교에 쓴 BGE-m3와 동일 패밀리
2. **Qwen3-Reranker-4B** (`Qwen/Qwen3-Reranker-4B`) — 다국어 범용, 2025년 최신, 임베딩 비교 1위였던 Qwen3-Embedding-4B와 짝을 이루는 리랭커 (yes/no 토큰 확률 기반 생성형 리랭커)
3. **ko-reranker** (`Dongjin-kr/ko-reranker`, ~560M) — 한국어 특화, bge-reranker-large를 한국어로 파인튜닝
4. **bge-reranker-v2-m3-ko** (`dragonkue/bge-reranker-v2-m3-ko`, ~568M) — 한국어 특화, bge-reranker-v2-m3를 한국어로 파인튜닝

4개 모두 순서대로 실행하면 되고, VRAM이 부족하면 Qwen3-Reranker 모델명을 `Qwen/Qwen3-Reranker-0.6B`로 바꾸세요.

**준비물**: 런타임 > 런타임 유형 변경 > GPU(T4 이상) 선택.


## 0. 공통 설치 및 데이터 준비

In [ ]:
!pip install -q -U sentence-transformers transformers accelerate scikit-learn


In [ ]:
import time
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("⚠️ GPU가 안 잡혔습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요.")

import transformers
print("transformers 버전:", transformers.__version__)


In [ ]:
import json
import os

# embedding_model_comparison.ipynb와 동일한 KOSIS 실제 통계표 1053개 코퍼스.
CORPUS_PATH = "kosis_large_corpus.json"
if not os.path.exists(CORPUS_PATH):
    raise FileNotFoundError(
        f"'{CORPUS_PATH}'가 없습니다. 이 노트북과 같은 폴더의 kosis_large_corpus.json 파일을 "
        "Colab 왼쪽 파일 탭에 업로드하세요 (파일 아이콘 > 업로드)."
    )
with open(CORPUS_PATH, encoding="utf-8") as f:
    TABLES = json.load(f)

# agent/mapping/test_mapping.py의 원래 12개 + 변별력 강화를 위해 38개 추가한 50개 테스트 케이스
# (문장, claim_type, 정답 table_id, 카테고리)
# embedding_model_comparison.ipynb와 동일한 벤치마크를 재사용해서 두 비교 결과를 나란히 볼 수 있게 함.
TEST_CASES = [
    # 고용/노동 (DT_1DA7001S) - 8개
    ("지난달 청년 실업률이 6%에 육박했다", "규모", "DT_1DA7001S", "고용/노동"),
    ("고용률이 역대 최고치를 기록했다", "비교", "DT_1DA7001S", "고용/노동"),
    ("취업자 수가 46개월 만에 감소 전환했다", "증감률", "DT_1DA7001S", "고용/노동"),
    ("일할 사람을 구하는 이들보다 놀고 있는 사람이 더 늘었다", "비교", "DT_1DA7001S", "고용/노동"),
    ("직장을 가진 국민의 비율이 소폭 상승했다", "증감률", "DT_1DA7001S", "고용/노동"),
    ("청년들이 일자리를 찾지 못해 거리로 나섰다", "규모", "DT_1DA7001S", "고용/노동"),
    ("경기 둔화로 기업들이 채용을 줄이면서 시장이 얼어붙었다", "비교", "DT_1DA7001S", "고용/노동"),
    ("은퇴 세대가 늘며 경제활동 참가율이 낮아졌다", "증감률", "DT_1DA7001S", "고용/노동"),

    # 물가/CPI (DT_1J22003) - 7개
    ("지난달 소비자물가가 전년 동월 대비 2.2% 올랐다", "증감률", "DT_1J22003", "물가/CPI"),
    ("생활물가가 5개월 연속 올랐다", "증감률", "DT_1J22003", "물가/CPI"),
    ("장바구니 물가가 서민 가계를 압박하고 있다", "규모", "DT_1J22003", "물가/CPI"),
    ("마트에서 장 보는 비용이 눈에 띄게 늘었다", "증감률", "DT_1J22003", "물가/CPI"),
    ("밥상 물가 부담이 커지며 소비 심리가 위축됐다", "규모", "DT_1J22003", "물가/CPI"),
    ("라면, 과자 등 가공식품 가격이 줄줄이 인상됐다", "규모", "DT_1J22003", "물가/CPI"),
    ("치솟는 물가에 서민들의 살림살이가 팍팍해졌다", "규모", "DT_1J22003", "물가/CPI"),

    # 인구 (DT_1B04005N) - 7개
    ("전국 주민등록인구가 5000만명 아래로 떨어졌다", "규모", "DT_1B04005N", "인구"),
    ("인구 감소가 가입자 이탈에 영향을 줬다", "비교", "DT_1B04005N", "인구"),
    ("우리나라에 사는 사람 수가 계속 줄어들고 있다", "비교", "DT_1B04005N", "인구"),
    ("지방 소멸 위기 속에 지역 인구가 급격히 줄고 있다", "비교", "DT_1B04005N", "인구"),
    ("빈집이 늘고 학교가 문을 닫는 등 인구 감소의 그림자가 짙어졌다", "전망", "DT_1B04005N", "인구"),
    ("전국 총인구가 사상 처음으로 감소세로 돌아섰다", "역대기록", "DT_1B04005N", "인구"),
    ("인구 순유출이 이어지며 지역 소멸 우려가 커지고 있다", "전망", "DT_1B04005N", "인구"),

    # 경제성장 (DT_200Y102) - 7개
    ("한국 경제성장률이 3개 분기 연속 0%대에 머물렀다", "비교", "DT_200Y102", "경제성장"),
    ("올해 경제성장률 전망치가 1.8%로 하향 조정됐다", "전망", "DT_200Y102", "경제성장"),
    ("나라 경제 전체 규모가 지난 분기보다 소폭 커졌다", "증감률", "DT_200Y102", "경제성장"),
    ("국내 경제가 눈에 띄게 활력을 잃어가고 있다", "비교", "DT_200Y102", "경제성장"),
    ("한국은행이 올해 성장 경로가 당초 예상보다 완만할 것이라고 밝혔다", "전망", "DT_200Y102", "경제성장"),
    ("내수 부진과 수출 둔화가 겹치며 경제 전반이 흔들렸다", "비교", "DT_200Y102", "경제성장"),
    ("우리 경제의 잠재성장률이 계속 낮아지고 있다는 우려가 나온다", "전망", "DT_200Y102", "경제성장"),

    # 무역/수출입 (DT_1R11006_FRM101) - 7개
    ("지난해 수출이 6838억달러로 역대 최대를 기록했다", "규모", "DT_1R11006_FRM101", "무역/수출입"),
    ("무역수지가 3년 만에 흑자로 전환했다", "비교", "DT_1R11006_FRM101", "무역/수출입"),
    ("해외로 내다 판 물건 값어치가 사상 최고를 찍었다", "역대기록", "DT_1R11006_FRM101", "무역/수출입"),
    ("반도체 훈풍에 힘입어 해외 판매가 늘었다", "증감률", "DT_1R11006_FRM101", "무역/수출입"),
    ("미국발 관세 여파로 대외 교역 여건이 악화됐다", "전망", "DT_1R11006_FRM101", "무역/수출입"),
    ("글로벌 경기 둔화가 국내 제조업 수출 전선에 그림자를 드리웠다", "전망", "DT_1R11006_FRM101", "무역/수출입"),
    ("대중국 수출이 부진하며 무역수지에 부담을 줬다", "비교", "DT_1R11006_FRM101", "무역/수출입"),

    # 부동산/주택 (DT_30404_B012) - 7개
    ("전국 집값이 하락세를 보였다", "비교", "DT_30404_B012", "부동산/주택"),
    ("서울 아파트값이 다시 오름세로 돌아섰다", "비교", "DT_30404_B012", "부동산/주택"),
    ("내 집 마련 비용 부담이 갈수록 커지고 있다", "증감률", "DT_30404_B012", "부동산/주택"),
    ("부동산 시장이 얼어붙으며 거래가 뚝 끊겼다", "규모", "DT_30404_B012", "부동산/주택"),
    ("강남 3구를 중심으로 매물이 자취를 감췄다", "규모", "DT_30404_B012", "부동산/주택"),
    ("고금리 장기화로 주택 수요가 크게 위축됐다", "비교", "DT_30404_B012", "부동산/주택"),
    ("전셋값도 동반 상승하며 주거비 부담이 커졌다", "증감률", "DT_30404_B012", "부동산/주택"),

    # 출생/사망/혼인 (DT_1B8000G) - 7개
    ("혼인 건수가 역대 최저를 기록했다", "역대기록", "DT_1B8000G", "출생/사망/혼인"),
    ("합계출산율이 0.7명대로 떨어졌다", "규모", "DT_1B8000G", "출생/사망/혼인"),
    ("아이 낳는 사람이 갈수록 줄어들고 있다", "비교", "DT_1B8000G", "출생/사망/혼인"),
    ("결혼하는 커플이 눈에 띄게 감소했다", "비교", "DT_1B8000G", "출생/사망/혼인"),
    ("저출생 고령화 여파로 학령인구가 급감했다", "비교", "DT_1B8000G", "출생/사망/혼인"),
    ("비혼과 만혼이 확산되며 가족 형태가 달라지고 있다", "전망", "DT_1B8000G", "출생/사망/혼인"),
    ("인구 절벽이라는 말이 나올 정도로 신생아 수가 급감했다", "비교", "DT_1B8000G", "출생/사망/혼인"),
]

doc_texts = [t["embedding_text"] for t in TABLES]
query_texts = [c[0] for c in TEST_CASES]
id_to_doc = {t["tblId"]: t["embedding_text"] for t in TABLES}
table_ids_all = [t["tblId"] for t in TABLES]
print(f"표 {len(TABLES)}개, 테스트 케이스 {len(TEST_CASES)}개 로드 완료")


---
## 1. 후보 shortlist 생성 (TF-IDF)

리랭커는 "1053개 표 중 무엇이 정답인가"를 처음부터 찾는 역할이 아니라, 이미 추려진 후보들을
재정렬하는 2단계 컴포넌트입니다. 그래서 문자 2~4-gram TF-IDF 코사인 유사도로 쿼리별 상위
`SHORTLIST_SIZE`개를 뽑아 "헷갈리는 후보 목록"을 만들고(실제 파이프라인의 keyword_search +
embedding_search가 후보를 압축해주는 역할을 모사), 정답 표가 여기 없으면 강제로 추가합니다
(안 그러면 애초에 리랭커가 정답을 볼 기회조차 없어서 "재정렬 능력" 자체를 평가할 수 없음).


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

SHORTLIST_SIZE = 20

tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
doc_tfidf = tfidf.fit_transform(doc_texts)
query_tfidf = tfidf.transform(query_texts)
tfidf_sims = (query_tfidf @ doc_tfidf.T).toarray()  # (n_query, n_doc), 이미 L2 정규화되어 코사인 유사도와 동일

SHORTLISTS = []  # 쿼리별 [(table_id, doc_text), ...] 후보 리스트
already_covered = 0
for i, (sentence, claim_type, expected_id, category) in enumerate(TEST_CASES):
    order = np.argsort(-tfidf_sims[i])[:SHORTLIST_SIZE]
    candidate_ids = [table_ids_all[idx] for idx in order]
    if expected_id in candidate_ids:
        already_covered += 1
    else:
        candidate_ids.append(expected_id)  # 정답 강제 포함
    SHORTLISTS.append([(tid, id_to_doc[tid]) for tid in candidate_ids])

sizes = [len(s) for s in SHORTLISTS]
print(f"쿼리 {len(SHORTLISTS)}개, shortlist 평균 크기 {np.mean(sizes):.1f}개")
print(f"TF-IDF top-{SHORTLIST_SIZE} 안에 정답이 이미 있던 쿼리: {already_covered}/{len(TEST_CASES)} "
      f"(나머지는 정답을 강제로 끼워 넣음)")


---
## 2. 평가 함수

`score_fn(query, documents) -> list[float]` 형태의 함수를 받아 `SHORTLISTS`를 재정렬하고
top-1 정확도 / Recall@5 / MRR / 평균 마진(1등-2등 점수 차)을 계산합니다.

embedding 비교와 달리 후보 집합이 쿼리마다 다르므로(고정된 코퍼스 전체가 아니라 shortlist),
"이 상위 후보들 안에서 정답을 얼마나 앞으로 끌어오는가"만 봅니다 — 실제 파이프라인에서
리랭커가 맡는 역할과 동일합니다.


In [ ]:
def evaluate_reranker(model_name: str, score_fn, k_list=(1, 5)) -> dict:
    """score_fn(query, documents)로 SHORTLISTS를 재정렬해 top-1/Recall@k/MRR/마진을 계산."""
    correct_at_k = {k: 0 for k in k_list}
    reciprocal_ranks = []
    margins = []
    rows = []

    for i, (sentence, claim_type, expected_id, category) in enumerate(TEST_CASES):
        candidates = SHORTLISTS[i]
        cand_ids = [c[0] for c in candidates]
        cand_docs = [c[1] for c in candidates]

        scores = np.asarray(score_fn(sentence, cand_docs))
        order = np.argsort(-scores)
        ranked_ids = [cand_ids[idx] for idx in order]
        ranked_scores = [float(scores[idx]) for idx in order]

        predicted_id = ranked_ids[0]
        ok = predicted_id == expected_id

        for k in k_list:
            if expected_id in ranked_ids[:k]:
                correct_at_k[k] += 1

        rank_of_correct = ranked_ids.index(expected_id) + 1 if expected_id in ranked_ids else None
        reciprocal_ranks.append(1.0 / rank_of_correct if rank_of_correct else 0.0)

        margin = ranked_scores[0] - ranked_scores[1] if len(ranked_scores) > 1 else 0.0
        margins.append(margin)

        rows.append((category, sentence, expected_id, predicted_id, ranked_scores[0], ok, rank_of_correct))

    total = len(TEST_CASES)
    mrr = float(np.mean(reciprocal_ranks))
    mean_margin = float(np.mean(margins))

    print()
    print(f"=== {model_name} ===")
    for k in k_list:
        print(f"  Recall@{k}: {correct_at_k[k]}/{total} ({correct_at_k[k] / total * 100:.1f}%)")
    print(f"  MRR: {mrr:.4f}")
    print(f"  평균 1등-2등 마진: {mean_margin:.4f}")
    print()
    for category, sentence, expected_id, predicted_id, score, ok, rank_of_correct in rows:
        mark = "O" if ok else "X"
        print(f"[{mark}] [{category}] {sentence}")
        print(f"      기대={expected_id} 예측={predicted_id} score={score:.3f} 정답순위={rank_of_correct}")

    return {
        "model": model_name,
        "correct": correct_at_k[1],
        "total": total,
        "accuracy": correct_at_k[1] / total,
        "recall_at_5": correct_at_k.get(5, correct_at_k[1]) / total,
        "mrr": mrr,
        "mean_margin": mean_margin,
    }


RESULTS = {}  # 모델명 -> evaluate_reranker() 반환값. 아래 섹션들을 실행할 때마다 채워짐.


---
## 1. BGE-reranker-v2-m3 (BAAI/bge-reranker-v2-m3, ~568M)

다국어 cross-encoder. `sentence-transformers`의 `CrossEncoder`로 로드하고, 원시 로짓에
시그모이드를 씌워 0~1 관련도 점수로 씁니다.


In [ ]:
from sentence_transformers import CrossEncoder

t0 = time.time()
bge_reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=device)


def bge_score_fn(query, documents):
    pairs = [[query, doc] for doc in documents]
    raw = np.asarray(bge_reranker.predict(pairs))
    return 1 / (1 + np.exp(-raw))


RESULTS["BGE-reranker-v2-m3"] = evaluate_reranker("BGE-reranker-v2-m3", bge_score_fn)
print(f"\n소요 시간: {time.time() - t0:.1f}s")

del bge_reranker
torch.cuda.empty_cache()


---
## 2. Qwen3-Reranker-4B (Qwen/Qwen3-Reranker-4B)

Qwen3-Reranker는 분류 헤드가 아니라 **생성형(decoder) 리랭커**입니다. "이 문서가 쿼리에
맞습니까? yes/no로만 답하라"는 프롬프트를 넣고, 마지막 토큰 위치에서 "yes"와 "no" 토큰의
로짓을 softmax해서 "yes" 확률을 관련도 점수로 씁니다 (Qwen3-Reranker 공식 사용법).
VRAM이 부족하면 모델명을 `Qwen/Qwen3-Reranker-0.6B`로 바꿔서 재실행하세요.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_RERANKER_NAME = "Qwen/Qwen3-Reranker-4B"  # VRAM 부족하면 "Qwen/Qwen3-Reranker-0.6B"로 교체
QWEN_INSTRUCTION = "Given a Korean news claim sentence, judge whether the KOSIS statistical table description is the correct match"
MAX_LENGTH = 4096

t0 = time.time()
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_RERANKER_NAME, padding_side="left")
qwen_reranker = AutoModelForCausalLM.from_pretrained(
    QWEN_RERANKER_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()

token_false_id = qwen_tokenizer.convert_tokens_to_ids("no")
token_true_id = qwen_tokenizer.convert_tokens_to_ids("yes")

PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
    "Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
)
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
prefix_tokens = qwen_tokenizer.encode(PREFIX, add_special_tokens=False)
suffix_tokens = qwen_tokenizer.encode(SUFFIX, add_special_tokens=False)


def qwen_format(query, doc):
    return f"<Instruct>: {QWEN_INSTRUCTION}\n<Query>: {query}\n<Document>: {doc}"


@torch.no_grad()
def qwen_score_fn(query, documents, batch_size=8):
    pairs = [qwen_format(query, doc) for doc in documents]
    all_scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        inputs = qwen_tokenizer(
            batch, padding=False, truncation="longest_first",
            return_attention_mask=False,
            max_length=MAX_LENGTH - len(prefix_tokens) - len(suffix_tokens),
        )
        for j, ids in enumerate(inputs["input_ids"]):
            inputs["input_ids"][j] = prefix_tokens + ids + suffix_tokens
        inputs = qwen_tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=MAX_LENGTH)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        logits = qwen_reranker(**inputs).logits[:, -1, :]
        true_vec = logits[:, token_true_id]
        false_vec = logits[:, token_false_id]
        stacked = torch.stack([false_vec, true_vec], dim=1)
        probs = torch.nn.functional.log_softmax(stacked, dim=1)
        all_scores.extend(probs[:, 1].exp().tolist())
    return all_scores


RESULTS["Qwen3-Reranker"] = evaluate_reranker("Qwen3-Reranker", qwen_score_fn)
print(f"\n소요 시간: {time.time() - t0:.1f}s")

del qwen_reranker
torch.cuda.empty_cache()


---
## 3. ko-reranker (Dongjin-kr/ko-reranker, ~560M)

한국어 전용으로 파인튜닝된 bge-reranker-large 계열 cross-encoder. BGE-reranker-v2-m3와
동일하게 `CrossEncoder` + 시그모이드로 씁니다.


In [ ]:
t0 = time.time()
ko_reranker = CrossEncoder("Dongjin-kr/ko-reranker", max_length=512, device=device)


def ko_reranker_score_fn(query, documents):
    pairs = [[query, doc] for doc in documents]
    raw = np.asarray(ko_reranker.predict(pairs))
    return 1 / (1 + np.exp(-raw))


RESULTS["ko-reranker"] = evaluate_reranker("ko-reranker", ko_reranker_score_fn)
print(f"\n소요 시간: {time.time() - t0:.1f}s")

del ko_reranker
torch.cuda.empty_cache()


---
## 4. bge-reranker-v2-m3-ko (dragonkue/bge-reranker-v2-m3-ko, ~568M)

BGE-reranker-v2-m3를 한국어 데이터로 추가 파인튜닝한 모델. 사용법은 동일합니다.


In [ ]:
t0 = time.time()
ko_bge_reranker = CrossEncoder("dragonkue/bge-reranker-v2-m3-ko", max_length=512, device=device)


def ko_bge_reranker_score_fn(query, documents):
    pairs = [[query, doc] for doc in documents]
    raw = np.asarray(ko_bge_reranker.predict(pairs))
    return 1 / (1 + np.exp(-raw))


RESULTS["bge-reranker-v2-m3-ko"] = evaluate_reranker("bge-reranker-v2-m3-ko", ko_bge_reranker_score_fn)
print(f"\n소요 시간: {time.time() - t0:.1f}s")

del ko_bge_reranker
torch.cuda.empty_cache()


---
## 5. 최종 비교


In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "모델": name,
        "정답": r["correct"],
        "전체": r["total"],
        "top-1 정확도": f"{r['accuracy'] * 100:.1f}%",
        "Recall@5": f"{r['recall_at_5'] * 100:.1f}%",
        "MRR": f"{r['mrr']:.3f}",
        "평균 마진(1등-2등)": f"{r['mean_margin']:.3f}",
    }
    for name, r in RESULTS.items()
])
summary
